In [2]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    GlobalAveragePooling2D,
    Dense,
    Dropout
)

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [3]:
train_df = pd.read_csv("../dataset/train_split.csv")
val_df = pd.read_csv("../dataset/val_split.csv")

class_names = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad",
    "surprise"
]

print("Jumlah training   :", len(train_df))
print("Jumlah validation :", len(val_df))

Jumlah training   : 21947
Jumlah validation : 5487


In [4]:
IMG_SIZE = (64, 64)
BATCH_SIZE = 32


def load_image(image_path, label):
    image = tf.io.read_file(image_path)

    image = tf.image.decode_jpeg(
        image,
        channels=1
    )

    image = tf.cast(
        image,
        tf.float32
    )

    # Grayscale 1 channel → RGB 3 channel
    image = tf.image.grayscale_to_rgb(image)

    # 48×48 → 64×64
    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    # Preprocessing khusus MobileNetV2
    image = preprocess_input(image)

    return image, label

In [5]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        train_df["image_path"].values,
        train_df["label_index"].values
    )
)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (
        val_df["image_path"].values,
        val_df["label_index"].values
    )
)

train_dataset = train_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_dataset = val_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_dataset = train_dataset.shuffle(
    len(train_df),
    seed=42
).batch(
    BATCH_SIZE
).prefetch(
    tf.data.AUTOTUNE
)

val_dataset = val_dataset.batch(
    BATCH_SIZE
).prefetch(
    tf.data.AUTOTUNE
)

In [6]:
images, labels = next(iter(train_dataset))

print("Shape images :", images.shape)
print("Shape labels :", labels.shape)
print("Nilai minimum:", tf.reduce_min(images).numpy())
print("Nilai maksimum:", tf.reduce_max(images).numpy())

Shape images : (32, 64, 64, 3)
Shape labels : (32,)
Nilai minimum: -1.0
Nilai maksimum: 1.0


In [7]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(64, 64, 3)
)

base_model.trainable = False

print("Jumlah layer base model:", len(base_model.layers))

C:\Users\BRT\AppData\Local\Temp\ipykernel_20000\30208243.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 10s 1us/step
Jumlah layer base model: 154


In [8]:
model = Sequential([
    Input(shape=(64, 64, 3)),

    base_model,

    GlobalAveragePooling2D(),

    Dense(
        128,
        activation="relu"
    ),

    Dropout(0.5),

    Dense(
        7,
        activation="softmax"
    )
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 2, 2, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [9]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [10]:
print("Base model trainable:", base_model.trainable)
print("Jumlah parameter:", model.count_params())

Base model trainable: False
Jumlah parameter: 2422855


In [11]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10
)

Epoch 1/10


d:\JamTerbang\DL\KlasifikasiEkspresi\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


686/686 ━━━━━━━━━━━━━━━━━━━━ 303s 429ms/step - accuracy: 0.3244 - loss: 1.7211 - val_accuracy: 0.3814 - val_loss: 1.5895
Epoch 2/10
686/686 ━━━━━━━━━━━━━━━━━━━━ 305s 438ms/step - accuracy: 0.3876 - loss: 1.5836 - val_accuracy: 0.4024 - val_loss: 1.5470
Epoch 3/10
686/686 ━━━━━━━━━━━━━━━━━━━━ 324s 466ms/step - accuracy: 0.4039 - loss: 1.5334 - val_accuracy: 0.4135 - val_loss: 1.5302
Epoch 4/10
686/686 ━━━━━━━━━━━━━━━━━━━━ 326s 469ms/step - accuracy: 0.4241 - loss: 1.4948 - val_accuracy: 0.4051 - val_loss: 1.5275
Epoch 5/10
686/686 ━━━━━━━━━━━━━━━━━━━━ 331s 476ms/step - accuracy: 0.4359 - loss: 1.4551 - val_accuracy: 0.4210 - val_loss: 1.5313
Epoch 6/10
686/686 ━━━━━━━━━━━━━━━━━━━━ 327s 470ms/step - accuracy: 0.4529 - loss: 1.4228 - val_accuracy: 0.4175 - val_loss: 1.5238
Epoch 7/10
686/686 ━━━━━━━━━━━━━━━━━━━━ 340s 490ms/step - accuracy: 0.4686 - loss: 1.3877 - val_accuracy: 0.4181 - val_loss: 1.5211
Epoch 8/10
686/686 ━━━━━━━━━━━━━━━━━━━━ 339s 488ms/step - accuracy: 0.4797 - loss: 1.35

In [12]:
# Unfreeze MobileNetV2
base_model.trainable = True

# Tetap bekukan layer-layer awal
fine_tune_at = 100

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print("Total layer:", len(base_model.layers))
print("Layer yang mulai di-fine-tune:", fine_tune_at)

print("Trainable layers:",
      sum(layer.trainable for layer in base_model.layers))

Total layer: 154
Layer yang mulai di-fine-tune: 100
Trainable layers: 54


In [13]:
print("Base model trainable:", base_model.trainable)

print(
    "Trainable parameters:",
    sum(tf.size(variable).numpy()
        for variable in model.trainable_variables)
)

Base model trainable: True
Trainable parameters: 2026311


In [14]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [15]:
history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5
)

Epoch 1/5
686/686 ━━━━━━━━━━━━━━━━━━━━ 556s 775ms/step - accuracy: 0.2518 - loss: 2.3113 - val_accuracy: 0.3408 - val_loss: 1.9240
Epoch 2/5
686/686 ━━━━━━━━━━━━━━━━━━━━ 529s 755ms/step - accuracy: 0.2784 - loss: 1.8944 - val_accuracy: 0.3332 - val_loss: 1.8079
Epoch 3/5
686/686 ━━━━━━━━━━━━━━━━━━━━ 536s 776ms/step - accuracy: 0.2923 - loss: 1.8007 - val_accuracy: 0.3352 - val_loss: 1.7237
Epoch 4/5
686/686 ━━━━━━━━━━━━━━━━━━━━ 532s 770ms/step - accuracy: 0.3104 - loss: 1.7312 - val_accuracy: 0.3370 - val_loss: 1.6859
Epoch 5/5
686/686 ━━━━━━━━━━━━━━━━━━━━ 536s 775ms/step - accuracy: 0.3269 - loss: 1.6934 - val_accuracy: 0.3486 - val_loss: 1.6605
